# SOXX 原始披露篮子成分轮换：早期身份预检

只读原始 SEC 来源结果，不读价格/收益。精确名称交集是身份线索，不能作为确认的进出或换手率。

In [1]:
import json,hashlib,unicodedata,re,statistics
from pathlib import Path
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/stage1_soxx_reported_basket_rotation_identity_preflight_20261023';plan=json.loads((card/'plan.json').read_text());assert plan['frozen'] and not (card/'run_state.json').exists()
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
state={'plan_sha256':sha(card/'plan.json'),'phase':'source_result_read_attempted','source_result_load_attempts':6,'external_HTTP_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
def norm(s):return re.sub(r'\s+',' ',unicodedata.normalize('NFKC',s).casefold()).strip()
reports=[];source_rows=[]
for spec in plan['sources']:
 path=root/spec['path'];assert sha(path)==spec['sha256'];source=json.loads(path.read_text());parts=source['reports'];source_rows.append({'path':spec['path'],'reports':len(parts),'sha256':spec['sha256']})
 for r in parts:
  equities=r['equities'];assert len(equities)==r['parsed_equity_rows']==30 and r['status'] in ('numeric_source_closure_passed','numeric_closure_passed','accepted_original_quarter_table','numeric_original_30_equity_value_closure') ,(r.get('accession'),r.get('status'))
  assert r['accession'] and r['report_date'] and r['accepted_at_utc'] and all(x['security_original'] and x['shares_original']>0 and x['fair_value_usd_original']>0 for x in equities)
  names=[norm(x['security_original']) for x in equities];collisions=len(names)-len(set(names))
  reports.append({'accession':r['accession'],'report_date':r['report_date'],'accepted_at_utc':r['accepted_at_utc'],'form':r['form'],'source_result':spec['path'],'names':names,'collisions':collisions})
assert len(reports)==31 and len({r['accession'] for r in reports})==31 and len({r['report_date'] for r in reports})==31
reports=sorted(reports,key=lambda r:r['report_date']);assert reports[0]['report_date']=='2011-01-31' and reports[-1]['report_date'].startswith('2018')
transitions=[]
for a,b in zip(reports,reports[1:]):
 prior=set(a['names']);next=set(b['names']);shared=sorted(prior&next)
 transitions.append({'prior_period':a['report_date'],'next_period':b['report_date'],'prior_accession':a['accession'],'next_accession':b['accession'],'exact_normalized_name_overlap':len(shared),'prior_only_names':sorted(prior-next),'next_only_names':sorted(next-prior),'prior_identity_collision':a['collisions'],'next_identity_collision':b['collisions']})
counts=[t['exact_normalized_name_overlap'] for t in transitions]
result={'decision':'legacy_reported_basket_turnover_identity_unqualified_source_only','plan_sha256':sha(card/'plan.json'),'source_groups':source_rows,'original_reports':len(reports),'first_report_period':reports[0]['report_date'],'last_report_period':reports[-1]['report_date'],'all_report_rows':len(reports)*30,'reports_with_name_collision':sum(r['collisions']>0 for r in reports),'adjacent_transitions':len(transitions),'exact_name_overlap_min':min(counts),'exact_name_overlap_median':float(statistics.median(counts)),'exact_name_overlap_max':max(counts),'transitions_overlap_below_25':sum(v<25 for v in counts),'total_prior_only_name_slots':sum(len(t['prior_only_names']) for t in transitions),'transition_details':transitions,'CUSIP_or_issuer_stable_identity_in_these_result_rows':False,'turnover_factor_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':result['decision'],'objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','market_outcome_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':plan['artifact_policy'],'limitations':['Normalized exact issuer names are not stable security identifiers; unmatched names may be aliases or corporate events.','No full2011-2026 PIT turnover vector, market result or Stage1 account.','Source filing result values are backed by their upstream original-source/NAS cards; this card adds no new raw filing bytes.']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
state['phase']='finished';(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');print({k:result[k] for k in ('decision','original_reports','adjacent_transitions','exact_name_overlap_min','exact_name_overlap_median','transitions_overlap_below_25','total_prior_only_name_slots')})

{'decision': 'legacy_reported_basket_turnover_identity_unqualified_source_only', 'original_reports': 31, 'adjacent_transitions': 30, 'exact_name_overlap_min': 9, 'exact_name_overlap_median': 23.0, 'transitions_overlap_below_25': 17, 'total_prior_only_name_slots': 277}


# SOXX 原始披露篮子轮换：名称身份预检未通过

只读六组已经按原始 SEC 文件与 NAS 归档核过的2011–2018普通股来源结果，共31份报告、930个正值证券行和30次相邻报告比较。没有市场行情、未来结果、模型或账户。本卡保留每个报告期、accession、原始名称和相邻差异；只做 Unicode 与空白规范化，不猜 CUSIP、ticker、公司别名或进出。

严格名称交集每对最少 **9/30**、中位数 **23/30**、最多 **29/30**；17/30次低于25，累计有277个“前报告独有名称”槽位。报告内没有规范化名称碰撞。差异清单显示许多对账失败来自 `(a)(b)`、`a,b` 等 SEC 行尾脚注的排版变化，并非可确认的证券退出/进入；2018 年还见 Broadcom、ASML 等名称/法律主体过渡。不能把 277 个名称槽位当作换手、现金机会或 SOXX 预测特征。

**决定：`legacy_reported_basket_turnover_identity_unqualified_source_only`。** 固定精确名称版本在身份门关闭，零收益读取、零拟合、零账户。下一步若继续此不同于静态集中度的机制，先另立只修原始脚注标记语法的来源版本，逐报告检查合并碰撞和仍不匹配的主体；在可审计的跨期证券身份与完整2011–2026 SEC生效时钟证明前，不计算成分轮换因子或接入 Stage1。
